
# Credit Risk Scorecard & Profit-Optimal Approval Strategy

## 1. Business Problem

A credit card lender must decide which customers to extend credit to. Extending credit to customers who later default causes losses. Refusing credit to customers who would have repaid loses revenue. What approval cutoff maximizes expected profit?

## 2. Objective

A model that estimates each customer’s probability of default (PD).
A transparent points-based scorecard built from that model.
A profit-optimal cutoff and a comparison against simple alternatives.
 
## 3. Target definition

**DEFAULT = 1:** the customer defaulted on payment in the month following the observation window (October 2005)
**DEFAULT = 0:** no default in that month

The dataset defines default as failure to pay in the next month. Banks usually use stricter definitions, such as 90+ days past due, within a 12-month window. A one-month window may overstate the roles of very recent delays.
    
## 4. Unit of analysis and population

One row = one cardholder. Population: 30,000 cardholders of a Taiwanese card issuer, with six months of history (April to September 2005).

## 5. Observation window and performance window

Observation window: April–September 2005. All features (bills, payments, delays) come from here.
Decision point: end of September 2005.
Performance window: October 2005. The target comes from here.

## 6. The “no future information” rule

Every feature must be something the lender would know on the decision date. A feature built using October data (or any information generated because the customer is already in trouble) is leakage and is not allowed.

## 7. Success criteria

**Ranking power:**	Gini in a typical range for this data (roughly 0.45–0.60)
**Separation:**	KS above about 0.35
**Calibration:**	Predicted PDs close to actual default rates
**Stability:**	PSI below 0.1 between train and test
**Business value:**	A profit-optimal cutoff that beats “approve everyone” and an accuracy-based cutoff


In [5]:
import subprocess

# 1. Fetch your installed packages
packages = subprocess.check_output(["pip", "freeze"]).decode("utf-8", errors="ignore")

# 2. Write them to requirements.txt using explicit UTF-8 encoding
with open("../requirements.txt", "w", encoding="utf-8") as f:
    f.write(packages)

print("requirements.txt successfully created in clean UTF-8 text!")


requirements.txt successfully created in clean UTF-8 text!


In [6]:
# The text constraints we want to put inside our .gitignore file
gitignore_content = """data/
.ipynb_checkpoints/
*.csv
"""

# Write the file directly to the root folder with explicit UTF-8 encoding
with open("../.gitignore", "w", encoding="utf-8") as f:
    f.write(gitignore_content)

print(".gitignore successfully created and configured!")

.gitignore successfully created and configured!


In [7]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
import sklearn, xgboost, lightgbm, shap, statsmodels, optbinning
print("All libraries imported successfully")

C:\Users\Vaidehi Kushwaha\.conda\envs\credit-risk\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All libraries imported successfully


In [8]:
df = pd.read_csv(r"C:/Users/Vaidehi Kushwaha/Documents/credit-risk-scorecard-profit-optimization/data/credit_card_default.csv")
print(df.shape)
df.head()

(30001, 25)


,Unnamed: 0,X1,X2,X3,X4,X5,X6,X7,X8,X9,...,X15,X16,X17,X18,X19,X20,X21,X22,X23,Y
0,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
1,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
2,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
3,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
4,4,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0


In [9]:
target = df.columns[-1]   # change this to the real target column name
print(df[target].value_counts(normalize=True))

Y
0                             0.778774
1                             0.221193
default payment next month    0.000033
Name: proportion, dtype: float64


In [10]:
df = pd.read_csv(r"C:/Users/Vaidehi Kushwaha/Documents/credit-risk-scorecard-profit-optimization/data/credit_card_default.csv", header=1)
print(df.shape)
df.head()

(30000, 25)


,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


In [16]:
# Check the available columns first
print(df.columns.tolist())

# Use the actual target column name in your file
target = "default payment next month"

print("Total customers:", len(df))
print("Target counts:")
print(df[target].value_counts().sort_index())

print("\nTarget proportions:")
print(df[target].value_counts(normalize=True).sort_index())

default_rate = df[target].mean() * 100
print(f"\nDefault rate: {default_rate:.2f}%")

['ID', 'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'default payment next month']
Total customers: 30000
Target counts:
default payment next month
0    23364
1     6636
Name: count, dtype: int64

Target proportions:
default payment next month
0    0.7788
1    0.2212
Name: proportion, dtype: float64

Default rate: 22.12%
